# Solar Filament Segmentation Challenge 2026 — Optimized YOLO26 Segmentation Baseline

This notebook keeps the competition-oriented YOLO26s segmentation pipeline while optimizing CPU/GPU utilization.

## Main optimizations

- **Exactly 4 dedicated CPU worker processes** for custom preprocessing/post-processing.
- Each worker is restricted to **one CPU thread/core**, targeting approximately **400% aggregate CPU** when all workers are busy.
- **Ultralytics `cache="disk"`** is enabled for YOLO training/validation data loading.
- Training uses `workers=4`.
- Validation overlaps **GPU inference with CPU metric calculation** using a persistent 4-worker pool.
- Submission post-processing and RLE encoding use the same 4-worker strategy.
- Validation inference uses **batched GPU inference** instead of one image per model call.
- Repeated COCO loading is avoided inside the main validation loop.
- CPU pools are created once and reused rather than repeatedly created per batch.
- The competition-oriented threshold is selected using **mean PQ**.

### Validation metrics

- Panoptic Quality (PQ)
- Segmentation Quality (SQ)
- Recognition Quality (RQ)
- Mean Dice
- Mean IoU
- Mean Precision / Recall
- TP / FP / FN
- One-to-many fragmentation
- Many-to-one over-merging

> Test-set ground truth is unavailable to this notebook, so test accuracy is never calculated.

## 1. Install dependencies and import libraries

Ultralytics supports `cache="disk"` for storing dataset images on disk and exposes a `workers` training argument for data loading. This notebook uses four workers for the custom CPU pipeline as well.

In [ ]:
%pip install -q -U ultralytics pycocotools scikit-learn pandas matplotlib tqdm torchmetrics threadpoolctl

In [ ]:
from pathlib import Path
import os
import random
import shutil
import time
import warnings
import multiprocessing
from concurrent.futures import ThreadPoolExecutor
from functools import partial

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from torchmetrics.segmentation import DiceScore
from threadpoolctl import threadpool_limits
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
YOLO_DEVICE = 0 if DEVICE.type == "cuda" else "cpu"

print(f"Training device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA devices available: {torch.cuda.device_count()}")

CPU_COUNT = os.cpu_count() or 1

print(f"Logical CPU count: {CPU_COUNT}")

## 2. Configure paths and experiment

The custom CPU pipeline deliberately uses **4 workers**. Each worker is constrained to one CPU thread. This avoids the common situation where four processes each start multiple OpenCV/BLAS threads and oversubscribe the CPU.

For YOLO training, `workers=4` and `cache="disk"` are used.

In [ ]:
SEED = 42

IMAGE_SIZE = 512
BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2

# Keep 1 for a smoke test. Increase for a real experiment.
NUM_EPOCHS = 1

VALIDATION_FRACTION = 0.20
PATIENCE = 10

# Dedicated CPU worker count requested for this experiment.
NUM_WORKERS = 4

# One worker thread per worker process.
WORKER_CPU_THREADS = 1

# Number of validation images sent to the GPU at once.
VAL_BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2

# Keep the number of queued CPU jobs bounded.
MAX_PENDING_VALIDATION = NUM_WORKERS * 4

MIN_COMPONENT_AREA = 20

YOLO_MODEL_NAME = "yolo26s-seg.pt"

YOLO_DATA_DIR = Path.cwd() / "yolo_filament_dataset"
RUNS_DIR = Path.cwd() / "yolo_runs"
RUN_NAME = "yolo26s_filament_optimized"

DATA_YAML = YOLO_DATA_DIR / "filament.yaml"

SUBMISSION_PATH = (
    Path.cwd()
    / "solar_filament_yolo26s_submission.csv"
)

FINAL_METRICS_PATH = (
    Path.cwd()
    / "solar_filament_yolo26s_validation_metrics.csv"
)

PER_IMAGE_METRICS_PATH = (
    Path.cwd()
    / "solar_filament_yolo26s_per_image_metrics.csv"
)

REBUILD_YOLO_DATASET = True
CREATE_SUBMISSION = True

CONFIDENCE_CANDIDATES = (
    0.15,
    0.25,
    0.35,
    0.45,
    0.55,
)

PQ_MATCH_IOU = 0.50


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_main_process_threads():
    cv2.setNumThreads(WORKER_CPU_THREADS)

    try:
        torch.set_num_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass


configure_main_process_threads()
set_seed()

print(f"Custom CPU workers: {NUM_WORKERS}")
print(
    f"Target aggregate CPU usage when all workers are busy: "
    f"~{NUM_WORKERS * 100}%"
)
print(f"GPU validation batch size: {VAL_BATCH_SIZE}")
print(f"Ultralytics training workers: {NUM_WORKERS}")
print("Ultralytics image cache mode: disk")

In [ ]:
def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/kaggle/input"),
    ]

    for parent in candidates:
        if not parent.exists():
            continue

        for annotation in parent.glob(
            "**/" + expected
        ):
            root = annotation.parent.parent

            if (
                (
                    root
                    / "train"
                    / "train_images"
                ).is_dir()
                and (
                    root
                    / "test"
                    / "test_images"
                ).is_dir()
            ):
                return root

    raise FileNotFoundError(
        "Could not find MAGFiLO_1.0_Kaggle_2026 "
        "with train and test image folders."
    )


DATA_ROOT = find_dataset_root()

TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"
TEST_IMAGE_DIR = (
    DATA_ROOT / "test" / "test_images"
)

ANNOTATION_PATH = (
    TRAIN_DIR
    / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

print(f"Dataset root: {DATA_ROOT}")
print(f"Annotation JSON: {ANNOTATION_PATH}")
print(f"YOLO model: {YOLO_MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(
    f"Validation fraction: "
    f"{VALIDATION_FRACTION:.0%}"
)
print(
    f"PQ matching IoU threshold: "
    f"{PQ_MATCH_IOU:.2f}"
)

## 3. Load annotations and create a leakage-safe split

The split is performed using the physical filename rather than the COCO record ID. Multiple annotation records for the same JPEG remain on one side of the split.

In [ ]:
coco = COCO(str(ANNOTATION_PATH))

image_ids = list(coco.imgs.keys())

records = pd.DataFrame(
    [
        {
            "image_id": image_id,
            "file_name": coco.imgs[image_id]["file_name"],
        }
        for image_id in image_ids
    ]
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
)

train_idx, val_idx = next(
    splitter.split(
        records,
        groups=records["file_name"],
    )
)

train_records = (
    records
    .iloc[train_idx]
    .reset_index(drop=True)
)

val_records = (
    records
    .iloc[val_idx]
    .reset_index(drop=True)
)

train_files = set(
    train_records["file_name"]
)

val_files = set(
    val_records["file_name"]
)

assert train_files.isdisjoint(
    val_files
), "A physical image appears in both splits."

print(
    f"Training records: "
    f"{len(train_records):,} "
    f"across {len(train_files):,} physical images"
)

print(
    f"Validation records: "
    f"{len(val_records):,} "
    f"across {len(val_files):,} physical images"
)

print(
    f"Train/validation filename overlap: "
    f"{len(train_files & val_files)}"
)

## 4. Build the YOLO segmentation dataset

COCO masks are converted to YOLO polygon labels.

The dataset-building pool also uses exactly four CPU workers. Each worker is restricted to one CPU thread.

In [ ]:
def configure_worker_process():
    cv2.setNumThreads(WORKER_CPU_THREADS)

    try:
        torch.set_num_threads(
            WORKER_CPU_THREADS
        )
    except RuntimeError:
        pass

    try:
        torch.set_num_interop_threads(
            WORKER_CPU_THREADS
        )
    except RuntimeError:
        pass

    try:
        threadpool_limits(
            limits=WORKER_CPU_THREADS
        )
    except Exception:
        pass

    if hasattr(os, "sched_getaffinity"):
        available_cpus = sorted(
            os.sched_getaffinity(0)
        )

        if available_cpus:
            identity = (
                multiprocessing.current_process()
                ._identity
            )

            worker_index = (
                identity[0] - 1
                if identity
                else 0
            )

            cpu = available_cpus[
                worker_index
                % len(available_cpus)
            ]

            if hasattr(os, "sched_setaffinity"):
                os.sched_setaffinity(
                    0,
                    {cpu},
                )


def mask_to_yolo_segments(
    mask,
    min_area,
):
    mask = (
        mask > 0
    ).astype(np.uint8)

    height, width = mask.shape

    count, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8,
        )
    )

    segments = []

    for label in range(
        1,
        count,
    ):
        area = int(
            stats[
                label,
                cv2.CC_STAT_AREA,
            ]
        )

        if area < min_area:
            continue

        component = (
            labels == label
        ).astype(np.uint8)

        contours, _ = (
            cv2.findContours(
                component,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE,
            )
        )

        if not contours:
            continue

        contour = max(
            contours,
            key=cv2.contourArea,
        )

        if (
            len(contour) < 3
            or cv2.contourArea(contour) <= 0
        ):
            continue

        points = (
            contour
            .reshape(-1, 2)
            .astype(np.float32)
        )

        points[:, 0] /= width
        points[:, 1] /= height

        points = np.clip(
            points,
            0.0,
            1.0,
        )

        if len(points) >= 3:
            segments.append(points)

    return segments


def worker_process_record(
    row_dict,
    annotation_path_str,
    min_area,
    train_image_dir_str,
    image_dir_str,
    label_dir_str,
):
    global _local_coco

    configure_worker_process()

    if "_local_coco" not in globals():
        import sys

        old_stdout = sys.stdout
        sys.stdout = open(
            os.devnull,
            "w",
        )

        _local_coco = COCO(
            annotation_path_str
        )

        sys.stdout = old_stdout

    train_img_dir = Path(
        train_image_dir_str
    )

    img_out_dir = Path(
        image_dir_str
    )

    lbl_out_dir = Path(
        label_dir_str
    )

    image_id = row_dict[
        "image_id"
    ]

    file_name = row_dict[
        "file_name"
    ]

    source_path = (
        train_img_dir
        / file_name
    )

    if not source_path.is_file():
        return 0

    source_stem = Path(
        file_name
    ).stem

    sample_stem = (
        f"{image_id}_{source_stem}"
    )

    image_path = (
        img_out_dir
        / f"{sample_stem}.jpg"
    )

    label_path = (
        lbl_out_dir
        / f"{sample_stem}.txt"
    )

    image_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        image_path.exists()
        or image_path.is_symlink()
    ):
        image_path.unlink()

    try:
        image_path.symlink_to(
            source_path.resolve()
        )
    except OSError:
        shutil.copy2(
            source_path,
            image_path,
        )

    annotations = (
        _local_coco.loadAnns(
            _local_coco.getAnnIds(
                imgIds=[image_id]
            )
        )
    )

    lines = []

    for annotation in annotations:
        mask = (
            _local_coco
            .annToMask(annotation)
            .astype(np.uint8)
        )

        for points in mask_to_yolo_segments(
            mask,
            min_area,
        ):
            flat = points.reshape(-1)

            values = ["0"]

            values.extend(
                f"{value:.6f}"
                for value in flat
            )

            lines.append(
                " ".join(values)
            )

    label_path.write_text(
        "\n".join(lines)
        + ("\n" if lines else ""),
        encoding="utf-8",
    )

    return len(lines)


def write_yolo_split_parallel(
    split_records,
    split_name,
):
    print(
        f"\n[START] Building "
        f"YOLO split: '{split_name}'",
        flush=True,
    )

    image_dir = (
        YOLO_DATA_DIR
        / "images"
        / split_name
    )

    label_dir = (
        YOLO_DATA_DIR
        / "labels"
        / split_name
    )

    image_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    label_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    records_list = (
        split_records[
            ["image_id", "file_name"]
        ]
        .to_dict(
            orient="records"
        )
    )

    worker_task = partial(
        worker_process_record,
        annotation_path_str=str(
            ANNOTATION_PATH
        ),
        min_area=MIN_COMPONENT_AREA,
        train_image_dir_str=str(
            TRAIN_IMAGE_DIR
        ),
        image_dir_str=str(
            image_dir
        ),
        label_dir_str=str(
            label_dir
        ),
    )

    empty_labels = 0
    total_instances = 0

    with multiprocessing.Pool(
        processes=NUM_WORKERS,
        initializer=configure_worker_process,
    ) as pool:

        results = tqdm(
            pool.imap_unordered(
                worker_task,
                records_list,
                chunksize=4,
            ),
            total=len(records_list),
            desc=(
                f"Preparing "
                f"YOLO {split_name}"
            ),
        )

        for idx, instance_count in enumerate(
            results,
            start=1,
        ):
            total_instances += (
                instance_count
            )

            if instance_count == 0:
                empty_labels += 1

            if (
                idx == 1
                or idx % 100 == 0
                or idx == len(records_list)
            ):
                print(
                    f"[{split_name.upper()}] "
                    f"{idx:,}/"
                    f"{len(records_list):,} "
                    f"| polygons: "
                    f"{total_instances:,} "
                    f"| empty: "
                    f"{empty_labels:,}",
                    flush=True,
                )

    print(
        f"[FINISHED] {split_name}: "
        f"{len(records_list):,} records | "
        f"{total_instances:,} polygons | "
        f"empty labels: "
        f"{empty_labels:,}\n",
        flush=True,
    )


if (
    REBUILD_YOLO_DATASET
    and YOLO_DATA_DIR.exists()
):
    print(
        f"[CLEANUP] Removing old directory: "
        f"{YOLO_DATA_DIR}",
        flush=True,
    )

    shutil.rmtree(
        YOLO_DATA_DIR
    )

write_yolo_split_parallel(
    train_records,
    "train",
)

write_yolo_split_parallel(
    val_records,
    "val",
)

DATA_YAML.write_text(
    (
        f"path: "
        f"'{YOLO_DATA_DIR.as_posix()}'\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: filament\n"
    ),
    encoding="utf-8",
)

print(
    f"YOLO dataset configuration: "
    f"{DATA_YAML}"
)

print(
    DATA_YAML.read_text()
)

## 5. Train YOLO26s-seg with disk caching

`cache="disk"` lets Ultralytics store dataset image arrays on disk instead of decoding the original JPEGs every epoch. `workers=4` controls the YOLO data-loader workers.

The four custom CPU workers used later are separate from these YOLO workers.

In [ ]:
print(
    "Building pretrained "
    "YOLO26s segmentation model..."
)

model = YOLO(
    YOLO_MODEL_NAME
)

training_start = (
    time.perf_counter()
)

train_results = model.train(
    data=str(DATA_YAML),

    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,

    device=YOLO_DEVICE,

    workers=NUM_WORKERS,
    cache="disk",

    patience=PATIENCE,

    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,

    pretrained=True,
    optimizer="auto",

    amp=(
        DEVICE.type == "cuda"
    ),

    seed=SEED,
    deterministic=True,

    fliplr=0.5,
    flipud=0.5,
    degrees=12.0,
    translate=0.04,
    scale=0.08,
    shear=0.0,
    perspective=0.0,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.15,

    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    plots=False,
    verbose=True,
)

training_time = (
    time.perf_counter()
    - training_start
)

try:
    run_dir = Path(
        model.trainer.save_dir
    )
except AttributeError:
    run_dir = (
        RUNS_DIR
        / RUN_NAME
    )

best_checkpoint = (
    run_dir
    / "weights"
    / "best.pt"
)

if not best_checkpoint.is_file():
    raise FileNotFoundError(
        "Best checkpoint was not created: "
        f"{best_checkpoint}"
    )

model = YOLO(
    str(best_checkpoint)
)

print(
    f"\n[SUCCESS] Training complete!\n"
    f"Run directory: {run_dir}\n"
    f"Best checkpoint: {best_checkpoint}\n"
    f"Training time: "
    f"{training_time / 60:.1f} min",
    flush=True,
)

## 6. Read training history

In [ ]:
history_path = (
    run_dir / "results.csv"
)

if not history_path.is_file():
    raise FileNotFoundError(
        f"YOLO training history not found: "
        f"{history_path}"
    )

history = pd.read_csv(
    history_path
)

history.columns = [
    column.strip()
    for column in history.columns
]

print(
    "Available history columns:"
)

print(
    ", ".join(history.columns)
)

epochs = np.arange(
    1,
    len(history) + 1,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
)

for column, label in [
    (
        "train/seg_loss",
        "Train segmentation loss",
    ),
    (
        "val/seg_loss",
        "Validation segmentation loss",
    ),
]:
    if column in history.columns:
        axes[0].plot(
            epochs,
            history[column],
            marker="o",
            label=label,
        )

axes[0].set(
    title="YOLO segmentation loss",
    xlabel="Epoch",
    ylabel="Loss",
)

axes[0].legend()
axes[0].grid(alpha=0.25)

for column, label in [
    (
        "metrics/mAP50(M)",
        "Mask mAP50",
    ),
    (
        "metrics/mAP50-95(M)",
        "Mask mAP50-95",
    ),
]:
    if column in history.columns:
        axes[1].plot(
            epochs,
            history[column],
            marker="o",
            label=label,
        )

axes[1].set(
    title="YOLO validation mask metrics",
    xlabel="Epoch",
    ylabel="Metric",
    ylim=(0, 1),
)

axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

print(
    f"Best checkpoint: "
    f"{best_checkpoint}"
)

## 7. Prediction and metric helpers

The prediction path is shared by validation and submission:

1. grayscale image → 3 identical pretrained-model channels
2. YOLO instance segmentation
3. confidence filtering
4. union of selected masks
5. connected-component splitting
6. evaluation or RLE encoding

In [ ]:
def to_yolo_image(image):
    if image.ndim != 2:
        raise ValueError(
            f"Expected grayscale image, "
            f"got shape {image.shape}"
        )

    return cv2.cvtColor(
        image,
        cv2.COLOR_GRAY2BGR,
    )


def get_test_image_paths():
    paths = [
        path
        for path in TEST_IMAGE_DIR.iterdir()
        if (
            path.is_file()
            and path.suffix.lower()
            in {".jpg", ".jpeg"}
        )
    ]

    return sorted(
        paths,
        key=lambda path: path.name,
    )


def predict_instances_batch(
    images,
    min_confidence=0.15,
):
    sources = [
        to_yolo_image(image)
        for image in images
    ]

    results = model.predict(
        source=sources,
        imgsz=IMAGE_SIZE,
        conf=min_confidence,
        iou=0.7,
        retina_masks=True,
        device=YOLO_DEVICE,
        batch=len(sources),
        verbose=False,
    )

    outputs = []

    for image, result in zip(
        images,
        results,
    ):
        if (
            result.masks is None
            or result.boxes is None
        ):
            outputs.append([])
            continue

        masks = (
            result.masks.data
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        confidences = (
            result.boxes.conf
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        native_height, native_width = (
            image.shape
        )

        instances = []

        for mask, confidence in zip(
            masks,
            confidences,
        ):
            if mask.shape != image.shape:
                mask = cv2.resize(
                    mask,
                    (
                        native_width,
                        native_height,
                    ),
                    interpolation=(
                        cv2.INTER_NEAREST
                    ),
                )

            instances.append(
                {
                    "mask": mask >= 0.5,
                    "confidence": float(
                        confidence
                    ),
                }
            )

        outputs.append(
            instances
        )

    return outputs


def predict_instances(
    image,
    min_confidence=0.15,
):
    return predict_instances_batch(
        [image],
        min_confidence=min_confidence,
    )[0]


def union_instances(
    instances,
    confidence_threshold,
):
    selected = [
        instance
        for instance in instances
        if instance["confidence"]
        >= confidence_threshold
    ]

    if not selected:
        return None

    foreground = np.zeros_like(
        selected[0]["mask"],
        dtype=bool,
    )

    for instance in selected:
        foreground |= instance["mask"]

    return foreground


def split_into_filaments(
    binary_mask,
    min_area=MIN_COMPONENT_AREA,
):
    count, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            binary_mask.astype(
                np.uint8
            ),
            connectivity=8,
        )
    )

    components = []

    for label in range(
        1,
        count,
    ):
        area = int(
            stats[
                label,
                cv2.CC_STAT_AREA,
            ]
        )

        if area < min_area:
            continue

        component = (
            labels == label
        )

        components.append(
            component
        )

    def component_key(
        component
    ):
        ys, xs = np.where(
            component
        )

        return (
            -int(component.sum()),
            int(ys.min())
            if len(ys)
            else 0,
            int(xs.min())
            if len(xs)
            else 0,
        )

    return sorted(
        components,
        key=component_key,
    )


def build_final_prediction_instances(
    instances,
    confidence_threshold,
):
    combined = union_instances(
        instances,
        confidence_threshold,
    )

    if combined is None:
        return []

    return split_into_filaments(
        combined,
        MIN_COMPONENT_AREA,
    )


def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(
        np.asfortranarray(
            binary_mask.astype(
                np.uint8
            )
        )
    )

    return encoded[
        "counts"
    ].decode("utf-8")

## 8. Competition-oriented validation

Validation is designed to keep all four CPU workers busy while the GPU continues performing inference.

### CPU workers handle

- COCO ground-truth mask reconstruction
- polygon rasterization
- connected components
- Dice / IoU
- PQ / SQ / RQ
- fragmentation / over-merging
- per-image metric aggregation

### GPU handles

- batched YOLO inference

A bounded asynchronous queue overlaps the two stages.

Each worker is limited to one thread/core. Thus four busy workers target approximately **400% aggregate CPU usage**.

In [ ]:
dice_metric = DiceScore(
    num_classes=2,
    include_background=False,
    average="micro",
    input_format="index",
)


def calculate_pixel_metrics(
    prediction,
    target,
):
    prediction_tensor = (
        torch.from_numpy(
            prediction.astype(
                np.int64
            )
        )
        .unsqueeze(0)
    )

    target_tensor = (
        torch.from_numpy(
            target.astype(
                np.int64
            )
        )
        .unsqueeze(0)
    )

    prediction_has_foreground = bool(
        prediction.any()
    )

    target_has_foreground = bool(
        target.any()
    )

    if (
        not prediction_has_foreground
        and not target_has_foreground
    ):
        dice = 1.0

    else:
        dice = float(
            dice_metric(
                prediction_tensor,
                target_tensor,
            ).item()
        )

        if np.isnan(dice):
            dice = 0.0

    intersection = int(
        np.logical_and(
            prediction,
            target,
        ).sum()
    )

    union = int(
        np.logical_or(
            prediction,
            target,
        ).sum()
    )

    true_positive = intersection

    false_positive = int(
        np.logical_and(
            prediction,
            ~target,
        ).sum()
    )

    false_negative = int(
        np.logical_and(
            ~prediction,
            target,
        ).sum()
    )

    iou = (
        intersection / union
        if union > 0
        else 1.0
    )

    precision_denominator = (
        true_positive
        + false_positive
    )

    recall_denominator = (
        true_positive
        + false_negative
    )

    precision = (
        true_positive
        / precision_denominator
        if precision_denominator > 0
        else 1.0
    )

    recall = (
        true_positive
        / recall_denominator
        if recall_denominator > 0
        else 1.0
    )

    return {
        "pixel_dice": float(
            dice
        ),
        "pixel_iou": float(
            iou
        ),
        "pixel_precision": float(
            precision
        ),
        "pixel_recall": float(
            recall
        ),
    }


def calculate_instance_metrics(
    prediction_instances,
    target_instances,
    iou_threshold=PQ_MATCH_IOU,
):
    num_predictions = len(
        prediction_instances
    )

    num_targets = len(
        target_instances
    )

    if (
        num_predictions == 0
        and num_targets == 0
    ):
        return {
            "pq": 1.0,
            "sq": 1.0,
            "rq": 1.0,
            "tp_instances": 0,
            "fp_instances": 0,
            "fn_instances": 0,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }

    if num_predictions == 0:
        return {
            "pq": 0.0,
            "sq": 0.0,
            "rq": 0.0,
            "tp_instances": 0,
            "fp_instances": 0,
            "fn_instances": num_targets,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }

    if num_targets == 0:
        return {
            "pq": 0.0,
            "sq": 0.0,
            "rq": 0.0,
            "tp_instances": 0,
            "fp_instances": num_predictions,
            "fn_instances": 0,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }

    iou_matrix = np.zeros(
        (
            num_targets,
            num_predictions,
        ),
        dtype=np.float32,
    )

    for gt_index, gt_mask in enumerate(
        target_instances
    ):
        gt_area = int(
            gt_mask.sum()
        )

        if gt_area == 0:
            continue

        for pred_index, pred_mask in enumerate(
            prediction_instances
        ):
            pred_area = int(
                pred_mask.sum()
            )

            if pred_area == 0:
                continue

            intersection = int(
                np.logical_and(
                    gt_mask,
                    pred_mask,
                ).sum()
            )

            if intersection == 0:
                continue

            union = (
                gt_area
                + pred_area
                - intersection
            )

            iou_matrix[
                gt_index,
                pred_index,
            ] = (
                intersection / union
            )

    gt_relation_counts = (
        iou_matrix
        > iou_threshold
    ).sum(axis=1)

    pred_relation_counts = (
        iou_matrix
        > iou_threshold
    ).sum(axis=0)

    one_to_many = int(
        (
            gt_relation_counts
            > 1
        ).sum()
    )

    many_to_one = int(
        (
            pred_relation_counts
            > 1
        ).sum()
    )

    candidate_pairs = []

    for gt_index in range(
        num_targets
    ):
        for pred_index in range(
            num_predictions
        ):
            iou = float(
                iou_matrix[
                    gt_index,
                    pred_index,
                ]
            )

            if iou > iou_threshold:
                candidate_pairs.append(
                    (
                        iou,
                        gt_index,
                        pred_index,
                    )
                )

    candidate_pairs.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    matched_gt = set()
    matched_predictions = set()
    matched_ious = []

    for (
        iou,
        gt_index,
        pred_index,
    ) in candidate_pairs:

        if gt_index in matched_gt:
            continue

        if (
            pred_index
            in matched_predictions
        ):
            continue

        matched_gt.add(
            gt_index
        )

        matched_predictions.add(
            pred_index
        )

        matched_ious.append(
            iou
        )

    tp = len(
        matched_ious
    )

    fp = (
        num_predictions - tp
    )

    fn = (
        num_targets - tp
    )

    matched_iou_sum = float(
        np.sum(
            matched_ious
        )
    )

    denominator = (
        tp
        + 0.5 * fp
        + 0.5 * fn
    )

    pq = (
        matched_iou_sum
        / denominator
        if denominator > 0
        else 0.0
    )

    sq = (
        matched_iou_sum / tp
        if tp > 0
        else 0.0
    )

    rq = (
        tp / denominator
        if denominator > 0
        else 0.0
    )

    return {
        "pq": float(pq),
        "sq": float(sq),
        "rq": float(rq),
        "tp_instances": tp,
        "fp_instances": fp,
        "fn_instances": fn,
        "one_to_many": one_to_many,
        "many_to_one": many_to_one,
        "matched_iou_sum": matched_iou_sum,
    }

In [ ]:
def load_validation_images(
    rows,
):
    images = []

    valid_rows = []

    for row in rows:
        image_path = (
            TRAIN_IMAGE_DIR
            / row["file_name"]
        )

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE,
        )

        if image is None:
            print(
                f"[WARNING] Could not read: "
                f"{image_path}",
                flush=True,
            )
            continue

        images.append(image)
        valid_rows.append(row)

    return valid_rows, images


def _worker_evaluate_validation_sample(
    task,
):
    configure_worker_process()

    global _local_val_coco

    if "_local_val_coco" not in globals():
        import sys

        old_stdout = sys.stdout
        sys.stdout = open(
            os.devnull,
            "w",
        )

        _local_val_coco = COCO(
            task["annotation_path"]
        )

        sys.stdout = old_stdout

    image_id = task[
        "image_id"
    ]

    file_name = task[
        "file_name"
    ]

    image_shape = tuple(
        task["image_shape"]
    )

    polygons = task[
        "polygons"
    ]

    confidences = task[
        "confidences"
    ]

    target_instances = []

    annotation_ids = (
        _local_val_coco.getAnnIds(
            imgIds=[image_id]
        )
    )

    annotations = (
        _local_val_coco.loadAnns(
            annotation_ids
        )
    )

    for annotation in annotations:
        mask = (
            _local_val_coco
            .annToMask(annotation)
            .astype(bool)
        )

        if mask.any():
            target_instances.append(
                mask
            )

    target_union = np.zeros(
        image_shape,
        dtype=bool,
    )

    for target_instance in (
        target_instances
    ):
        target_union |= (
            target_instance
        )

    outputs = {}

    for threshold in (
        CONFIDENCE_CANDIDATES
    ):
        prediction_union = np.zeros(
            image_shape,
            dtype=np.uint8,
        )

        selected_predictions = 0

        for polygon, confidence in zip(
            polygons,
            confidences,
        ):
            if (
                confidence
                < threshold
            ):
                continue

            polygon = np.asarray(
                polygon,
                dtype=np.float32,
            )

            if len(polygon) < 3:
                continue

            cv2.fillPoly(
                prediction_union,
                [
                    np.round(
                        polygon
                    ).astype(
                        np.int32
                    )
                ],
                1,
            )

            selected_predictions += 1

        prediction_union = (
            prediction_union
            .astype(bool)
        )

        prediction_instances = (
            split_into_filaments(
                prediction_union,
                MIN_COMPONENT_AREA,
            )
        )

        pixel_metrics = (
            calculate_pixel_metrics(
                prediction_union,
                target_union,
            )
        )

        instance_metrics = (
            calculate_instance_metrics(
                prediction_instances,
                target_instances,
            )
        )

        outputs[threshold] = {
            "file_name": file_name,

            "pixel_dice": pixel_metrics[
                "pixel_dice"
            ],

            "pixel_iou": pixel_metrics[
                "pixel_iou"
            ],

            "pixel_precision": pixel_metrics[
                "pixel_precision"
            ],

            "pixel_recall": pixel_metrics[
                "pixel_recall"
            ],

            "pq": instance_metrics[
                "pq"
            ],

            "sq": instance_metrics[
                "sq"
            ],

            "rq": instance_metrics[
                "rq"
            ],

            "tp_instances": instance_metrics[
                "tp_instances"
            ],

            "fp_instances": instance_metrics[
                "fp_instances"
            ],

            "fn_instances": instance_metrics[
                "fn_instances"
            ],

            "predicted_instances": len(
                prediction_instances
            ),

            "ground_truth_instances": len(
                target_instances
            ),

            "one_to_many": instance_metrics[
                "one_to_many"
            ],

            "many_to_one": instance_metrics[
                "many_to_one"
            ],

            "selected_predictions": (
                selected_predictions
            ),
        }

    return outputs

In [ ]:
validation_start = (
    time.perf_counter()
)

minimum_confidence = min(
    CONFIDENCE_CANDIDATES
)

validation_examples = []

threshold_results = {
    threshold: {
        "per_image": []
    }
    for threshold
    in CONFIDENCE_CANDIDATES
}

val_rows = (
    val_records[
        [
            "image_id",
            "file_name",
        ]
    ]
    .to_dict(
        orient="records"
    )
)

total_validation_records = len(
    val_rows
)

print(
    "\n"
    + "=" * 90
)

print(
    "                 PARALLEL VALIDATION"
)

print(
    "=" * 90
)

print(
    f"Validation records : "
    f"{total_validation_records:,}"
)

print(
    f"CPU workers        : "
    f"{NUM_WORKERS}"
)

print(
    f"Target aggregate CPU usage: "
    f"~{NUM_WORKERS * 100}%"
)

print(
    f"GPU batch size     : "
    f"{VAL_BATCH_SIZE}"
)

print(
    "CPU workload       : "
    "polygon rasterization + "
    "connected components + "
    "metrics"
)

print(
    "=" * 90,
    flush=True,
)

# Persistent CPU pool.
pool = multiprocessing.Pool(
    processes=NUM_WORKERS,
    initializer=configure_worker_process,
)

pending = []

try:
    for batch_start in range(
        0,
        total_validation_records,
        VAL_BATCH_SIZE,
    ):
        batch_rows = val_rows[
            batch_start:
            batch_start
            + VAL_BATCH_SIZE
        ]

        loaded_rows, images = (
            load_validation_images(
                batch_rows
            )
        )

        if not images:
            continue

        # ------------------------------------------------------
        # GPU inference in batches.
        # ------------------------------------------------------

        batch_instances = (
            predict_instances_batch(
                images,
                min_confidence=(
                    minimum_confidence
                ),
            )
        )

        # ------------------------------------------------------
        # Submit CPU evaluation asynchronously.
        # ------------------------------------------------------

        for row, image, instances in zip(
            loaded_rows,
            images,
            batch_instances,
        ):
            polygons = []
            confidences = []

            for instance in instances:
                mask = instance[
                    "mask"
                ]

                contours, _ = (
                    cv2.findContours(
                        mask.astype(
                            np.uint8
                        ),
                        cv2.RETR_EXTERNAL,
                        cv2.CHAIN_APPROX_SIMPLE,
                    )
                )

                if not contours:
                    continue

                contour = max(
                    contours,
                    key=cv2.contourArea,
                )

                if len(contour) < 3:
                    continue

                polygon = (
                    contour
                    .reshape(-1, 2)
                    .astype(
                        np.float32
                    )
                )

                polygons.append(
                    polygon
                )

                confidences.append(
                    float(
                        instance[
                            "confidence"
                        ]
                    )
                )

            task = {
                "annotation_path": str(
                    ANNOTATION_PATH
                ),
                "image_id": row[
                    "image_id"
                ],
                "file_name": row[
                    "file_name"
                ],
                "image_shape": (
                    image.shape
                ),
                "polygons": polygons,
                "confidences": confidences,
            }

            pending.append(
                pool.apply_async(
                    _worker_evaluate_validation_sample,
                    (task,),
                )
            )

            if len(pending) >= (
                MAX_PENDING_VALIDATION
            ):
                result = (
                    pending.pop(0).get()
                )

                for threshold in (
                    CONFIDENCE_CANDIDATES
                ):
                    threshold_results[
                        threshold
                    ][
                        "per_image"
                    ].append(
                        result[
                            threshold
                        ]
                    )

        # ------------------------------------------------------
        # Save a few examples for visualization.
        # ------------------------------------------------------

        for row, image, instances in zip(
            loaded_rows,
            images,
            batch_instances,
        ):
            if (
                len(validation_examples)
                >= 4
            ):
                break

            annotations = (
                coco.loadAnns(
                    coco.getAnnIds(
                        imgIds=[
                            row["image_id"]
                        ]
                    )
                )
            )

            target = np.zeros(
                image.shape,
                dtype=bool,
            )

            for annotation in (
                annotations
            ):
                target |= (
                    coco
                    .annToMask(
                        annotation
                    )
                    .astype(bool)
                )

            validation_examples.append(
                {
                    "image": image,
                    "target": target,
                    "instances": instances,
                    "file_name": row[
                        "file_name"
                    ],
                }
            )

        processed = min(
            batch_start
            + len(loaded_rows),
            total_validation_records,
        )

        print(
            f"[GPU + CPU] "
            f"Submitted/evaluated "
            f"{processed:,}/"
            f"{total_validation_records:,} "
            f"records | "
            f"pending CPU jobs: "
            f"{len(pending)}",
            flush=True,
        )

    print(
        "\n[CPU] Draining remaining "
        "validation workers...",
        flush=True,
    )

    while pending:
        result = (
            pending.pop(0).get()
        )

        for threshold in (
            CONFIDENCE_CANDIDATES
        ):
            threshold_results[
                threshold
            ][
                "per_image"
            ].append(
                result[
                    threshold
                ]
            )

finally:
    pool.close()
    pool.join()

validation_time = (
    time.perf_counter()
    - validation_start
)

threshold_rows = []

for threshold in (
    CONFIDENCE_CANDIDATES
):
    per_image_df = pd.DataFrame(
        threshold_results[
            threshold
        ]["per_image"]
    )

    if per_image_df.empty:
        continue

    threshold_rows.append(
        {
            "confidence": threshold,

            "PQ": per_image_df[
                "pq"
            ].mean(),

            "SQ": per_image_df[
                "sq"
            ].mean(),

            "RQ": per_image_df[
                "rq"
            ].mean(),

            "Mean Dice": per_image_df[
                "pixel_dice"
            ].mean(),

            "Mean IoU": per_image_df[
                "pixel_iou"
            ].mean(),

            "Mean Precision": (
                per_image_df[
                    "pixel_precision"
                ].mean()
            ),

            "Mean Recall": (
                per_image_df[
                    "pixel_recall"
                ].mean()
            ),

            "TP": int(
                per_image_df[
                    "tp_instances"
                ].sum()
            ),

            "FP": int(
                per_image_df[
                    "fp_instances"
                ].sum()
            ),

            "FN": int(
                per_image_df[
                    "fn_instances"
                ].sum()
            ),

            "Predicted Instances": int(
                per_image_df[
                    "predicted_instances"
                ].sum()
            ),

            "GT Instances": int(
                per_image_df[
                    "ground_truth_instances"
                ].sum()
            ),

            "One-to-Many": int(
                per_image_df[
                    "one_to_many"
                ].sum()
            ),

            "Many-to-One": int(
                per_image_df[
                    "many_to_one"
                ].sum()
            ),
        }
    )

threshold_summary = (
    pd.DataFrame(
        threshold_rows
    )
    .sort_values(
        "PQ",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

BEST_CONFIDENCE = float(
    threshold_summary.iloc[0][
        "confidence"
    ]
)

print(
    "\n"
    + "=" * 100
)

print(
    "                    VALIDATION RESULTS"
)

print(
    "=" * 100
)

display(
    threshold_summary.round(4)
)

print(
    f"\nSelected confidence threshold: "
    f"{BEST_CONFIDENCE:.2f}"
)

print(
    f"Best validation PQ: "
    f"{threshold_summary.iloc[0]['PQ']:.4f}"
)

print(
    f"Validation time: "
    f"{validation_time:.1f}s"
)

print(
    "=" * 100,
    flush=True,
)

## 9. Generate the competition submission

Test inference is also batched on the GPU. CPU connected-component extraction and RLE encoding are delegated to the same four-worker CPU strategy.

The final output remains native-resolution RLE.

In [ ]:
def _submission_worker(
    task
):
    configure_worker_process()

    combined_mask = task[
        "combined_mask"
    ]

    path_stem = task[
        "path_stem"
    ]

    components = (
        split_into_filaments(
            combined_mask,
            MIN_COMPONENT_AREA,
        )
    )

    rows = []

    for filament_counter, component in enumerate(
        components,
        start=1,
    ):
        rows.append(
            {
                "filament_id": (
                    f"{path_stem}_"
                    f"{filament_counter}"
                ),
                "segmentation_rle": (
                    mask_to_rle(
                        component
                    )
                ),
            }
        )

    return rows


def create_submission_parallel(
    output_path=SUBMISSION_PATH,
):
    paths = get_test_image_paths()

    if not paths:
        raise FileNotFoundError(
            f"No .jpg/.jpeg images found "
            f"in {TEST_IMAGE_DIR}"
        )

    total_images = len(
        paths
    )

    start_time = (
        time.perf_counter()
    )

    print(
        f"[START] Test images: "
        f"{total_images:,}",
        flush=True,
    )

    print(
        f"[CONFIG] GPU batch: "
        f"{VAL_BATCH_SIZE}",
        flush=True,
    )

    print(
        f"[CONFIG] CPU workers: "
        f"{NUM_WORKERS}",
        flush=True,
    )

    cpu_pool = multiprocessing.Pool(
        processes=NUM_WORKERS,
        initializer=configure_worker_process,
    )

    pending = []
    rows = []

    try:
        for batch_start in range(
            0,
            total_images,
            VAL_BATCH_SIZE,
        ):
            batch_paths = paths[
                batch_start:
                batch_start
                + VAL_BATCH_SIZE
            ]

            images = []

            for path in batch_paths:
                image = cv2.imread(
                    str(path),
                    cv2.IMREAD_GRAYSCALE,
                )

                if image is None:
                    raise OSError(
                        f"Could not read test image: "
                        f"{path}"
                    )

                images.append(image)

            batch_instances = (
                predict_instances_batch(
                    images,
                    min_confidence=(
                        BEST_CONFIDENCE
                    ),
                )
            )

            for path, instances in zip(
                batch_paths,
                batch_instances,
            ):
                combined = (
                    union_instances(
                        instances,
                        BEST_CONFIDENCE,
                    )
                )

                if combined is None:
                    combined = np.zeros(
                        images[0].shape,
                        dtype=bool,
                    )

                pending.append(
                    cpu_pool.apply_async(
                        _submission_worker,
                        (
                            {
                                "combined_mask": combined,
                                "path_stem": path.stem,
                            },
                        ),
                    )
                )

                if len(pending) >= (
                    MAX_PENDING_VALIDATION
                ):
                    rows.extend(
                        pending.pop(0).get()
                    )

            processed = min(
                batch_start
                + len(batch_paths),
                total_images,
            )

            print(
                f"[GPU + CPU] "
                f"Processed "
                f"{processed:,}/"
                f"{total_images:,}",
                f"| pending: "
                f"{len(pending)}",
                flush=True,
            )

        while pending:
            rows.extend(
                pending.pop(0).get()
            )

    finally:
        cpu_pool.close()
        cpu_pool.join()

    submission = pd.DataFrame(
        rows,
        columns=[
            "filament_id",
            "segmentation_rle",
        ],
    )

    if (
        not submission.empty
        and submission[
            "filament_id"
        ].duplicated().any()
    ):
        raise ValueError(
            "Generated submission "
            "contains duplicate filament IDs."
        )

    submission.to_csv(
        output_path,
        index=False,
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    print(
        f"\n[SUCCESS] Submission complete!\n"
        f"Processed: "
        f"{total_images:,} images\n"
        f"Filament rows: "
        f"{len(submission):,}\n"
        f"Runtime: "
        f"{elapsed / 60:.2f} min\n"
        f"Saved: "
        f"{output_path}",
        flush=True,
    )

    return submission


if CREATE_SUBMISSION:
    submission_df = (
        create_submission_parallel()
    )

    display(
        submission_df.head()
    )
else:
    submission_df = None

    print(
        "CREATE_SUBMISSION=False; "
        "submission generation skipped."
    )

## 10. Submission audit and final validation report

In [ ]:
if CREATE_SUBMISSION:
    audit_df = pd.read_csv(
        SUBMISSION_PATH
    )
else:
    audit_df = pd.DataFrame(
        columns=[
            "filament_id",
            "segmentation_rle",
        ]
    )

print(
    "--- Submission File Integrity Audit ---"
)

print(
    f"Total rows: "
    f"{len(audit_df):,}"
)

print(
    f"Null values: "
    f"{audit_df.isnull().sum().sum():,}"
)

duplicate_ids = 0

if "filament_id" in audit_df.columns:
    duplicate_ids = int(
        audit_df[
            "filament_id"
        ].duplicated().sum()
    )

print(
    f"Duplicate filament IDs: "
    f"{duplicate_ids:,}"
)

if "segmentation_rle" in audit_df.columns:
    empty_rle_count = int(
        audit_df[
            "segmentation_rle"
        ]
        .fillna("")
        .astype(str)
        .str.len()
        .eq(0)
        .sum()
    )
else:
    empty_rle_count = 0

print(
    f"Empty RLE strings: "
    f"{empty_rle_count:,}"
)

sample_rle_valid = False

if len(audit_df) > 0:
    sample_rle = (
        audit_df.iloc[0][
            "segmentation_rle"
        ]
    )

    sample_rle_valid = (
        isinstance(
            sample_rle,
            str,
        )
        and len(sample_rle) > 0
    )

print(
    f"Sample RLE check: "
    f"{sample_rle_valid}"
)

submission_valid = (
    set(audit_df.columns)
    == {
        "filament_id",
        "segmentation_rle",
    }
    and audit_df.isnull().sum().sum()
    == 0
    and duplicate_ids == 0
    and empty_rle_count == 0
    and (
        sample_rle_valid
        or len(audit_df) == 0
    )
)

print(
    "\n"
    + (
        "[VERIFIED] Submission structure "
        "passed the local audit."
        if submission_valid
        else "[WARNING] Submission structure "
             "needs inspection."
    )
)

best_row = (
    threshold_summary[
        threshold_summary[
            "confidence"
        ]
        == BEST_CONFIDENCE
    ]
    .iloc[0]
)

final_metrics = pd.DataFrame(
    {
        "Metric": [
            "Panoptic Quality (PQ)",
            "Segmentation Quality (SQ)",
            "Recognition Quality (RQ)",
            "Mean Dice",
            "Mean IoU",
            "Mean Precision",
            "Mean Recall",
            "True Positives",
            "False Positives",
            "False Negatives",
            "Predicted Instances",
            "Ground-Truth Instances",
            "One-to-Many (Fragmentation)",
            "Many-to-One (Over-Merging)",
            "Selected Confidence",
        ],
        "Value": [
            best_row["PQ"],
            best_row["SQ"],
            best_row["RQ"],
            best_row["Mean Dice"],
            best_row["Mean IoU"],
            best_row["Mean Precision"],
            best_row["Mean Recall"],
            int(best_row["TP"]),
            int(best_row["FP"]),
            int(best_row["FN"]),
            int(
                best_row[
                    "Predicted Instances"
                ]
            ),
            int(
                best_row[
                    "GT Instances"
                ]
            ),
            int(
                best_row[
                    "One-to-Many"
                ]
            ),
            int(
                best_row[
                    "Many-to-One"
                ]
            ),
            BEST_CONFIDENCE,
        ],
    }
)

print(
    "\n"
    + "=" * 75
)

print(
    "                 FINAL VALIDATION REPORT"
)

print(
    "=" * 75
)

display(
    final_metrics.round(4)
)

selected_per_image = pd.DataFrame(
    threshold_results[
        BEST_CONFIDENCE
    ]["per_image"]
)

distribution = (
    selected_per_image[
        [
            "pixel_dice",
            "pixel_iou",
            "pq",
            "sq",
            "rq",
        ]
    ]
    .agg(
        [
            "mean",
            "median",
            "std",
            "min",
            "max",
        ]
    )
    .T
)

print(
    "\n--- Per-image metric distribution ---"
)

display(
    distribution.round(4)
)

best_mask_map50 = (
    float(
        history[
            "metrics/mAP50(M)"
        ].max()
    )
    if "metrics/mAP50(M)"
    in history.columns
    else np.nan
)

best_mask_map50_95 = (
    float(
        history[
            "metrics/mAP50-95(M)"
        ].max()
    )
    if "metrics/mAP50-95(M)"
    in history.columns
    else np.nan
)

diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "YOLO best mask mAP50",
            "YOLO best mask mAP50-95",
            "Validation records",
            "Submission rows",
            "Submission valid",
        ],
        "Value": [
            best_mask_map50,
            best_mask_map50_95,
            len(selected_per_image),
            len(audit_df),
            submission_valid,
        ],
    }
)

print(
    "\n--- Model / submission diagnostics ---"
)

display(
    diagnostics
)

final_metrics.to_csv(
    FINAL_METRICS_PATH,
    index=False,
)

selected_per_image.to_csv(
    PER_IMAGE_METRICS_PATH,
    index=False,
)

print(
    f"Saved validation summary: "
    f"{FINAL_METRICS_PATH}"
)

print(
    f"Saved per-image metrics: "
    f"{PER_IMAGE_METRICS_PATH}"
)

## 11. Threshold visualization and validation examples

Threshold selection is based on **PQ**.

In [ ]:
fig, axis = plt.subplots(
    figsize=(7, 4.5)
)

axis.plot(
    threshold_summary[
        "confidence"
    ],
    threshold_summary[
        "PQ"
    ],
    marker="o",
    label="Validation PQ",
)

axis.axvline(
    BEST_CONFIDENCE,
    linestyle="--",
    label=(
        f"Selected: "
        f"{BEST_CONFIDENCE:.2f}"
    ),
)

axis.set(
    title=(
        "Validation PQ "
        "vs confidence threshold"
    ),
    xlabel=(
        "YOLO instance confidence"
    ),
    ylabel=(
        "Panoptic Quality (PQ)"
    ),
    ylim=(0, 1),
)

axis.grid(alpha=0.25)
axis.legend()

fig.tight_layout()
plt.show()


for example in validation_examples:
    image = example["image"]
    target = example["target"]
    instances = example["instances"]

    prediction = union_instances(
        instances,
        BEST_CONFIDENCE,
    )

    if prediction is None:
        prediction = np.zeros_like(
            target,
            dtype=bool,
        )

    selected_prediction_instances = (
        build_final_prediction_instances(
            instances,
            BEST_CONFIDENCE,
        )
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(14, 4.5),
    )

    axes[0].imshow(
        image,
        cmap="gray",
    )

    axes[0].set_title(
        "Input image"
    )

    axes[1].imshow(
        target,
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    axes[1].set_title(
        "Ground truth"
    )

    axes[2].imshow(
        prediction,
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    axes[2].set_title(
        "Prediction · "
        f"{len(selected_prediction_instances)} "
        "instance(s)"
    )

    for axis in axes:
        axis.axis("off")

    fig.suptitle(
        example["file_name"]
    )

    fig.tight_layout()
    plt.show()

## 12. Display a labeled prediction on an unseen test image

The test image is used only for inference/visualization. No test ground truth is accessed.

In [ ]:
def plot_labeled_prediction(
    image,
    instances,
    title,
):
    combined = union_instances(
        instances,
        BEST_CONFIDENCE,
    )

    components = (
        split_into_filaments(
            combined
        )
        if combined is not None
        else []
    )

    fig, axis = plt.subplots(
        figsize=(9, 9)
    )

    axis.imshow(
        image,
        cmap="gray",
    )

    for filament_index, component in enumerate(
        components,
        start=1,
    ):
        axis.contour(
            component.astype(
                np.uint8
            ),
            levels=[0.5],
            linewidths=1.2,
        )

        moments = cv2.moments(
            component.astype(
                np.uint8
            )
        )

        if moments["m00"]:
            center_x = (
                moments["m10"]
                / moments["m00"]
            )

            center_y = (
                moments["m01"]
                / moments["m00"]
            )
        else:
            ys, xs = np.where(
                component
            )

            center_x = xs.mean()
            center_y = ys.mean()

        area = int(
            component.sum()
        )

        axis.text(
            center_x,
            center_y,
            (
                f"F{filament_index}\n"
                f"{area:,} px"
            ),
            fontsize=8,
            ha="center",
            va="center",
            bbox=dict(
                boxstyle=(
                    "round,pad=0.2"
                ),
                facecolor="white",
                alpha=0.75,
            ),
        )

    axis.set_title(
        f"{title} · "
        f"{len(components)} "
        "predicted filament(s)"
    )

    axis.axis("off")

    fig.tight_layout()
    plt.show()


test_paths = (
    get_test_image_paths()
)

if not test_paths:
    raise FileNotFoundError(
        "No test images were found."
    )

test_path = test_paths[0]

test_image = cv2.imread(
    str(test_path),
    cv2.IMREAD_GRAYSCALE,
)

if test_image is None:
    raise OSError(
        f"Could not read test image: "
        f"{test_path}"
    )

test_instances = predict_instances(
    test_image,
    min_confidence=BEST_CONFIDENCE,
)

print(
    "Displaying labeled prediction "
    f"for test image: {test_path.name}"
)

plot_labeled_prediction(
    test_image,
    test_instances,
    (
        f"Test prediction · "
        f"{test_path.name} · "
        f"confidence "
        f"{BEST_CONFIDENCE:.2f}"
    ),
)

## 13. Experiment summary

### Main outputs

- `yolo_filament_dataset/` — temporary YOLO-format dataset
- `yolo_runs/yolo26s_filament_optimized/weights/best.pt` — best checkpoint
- `solar_filament_yolo26s_submission.csv` — competition submission
- `solar_filament_yolo26s_validation_metrics.csv` — validation summary
- `solar_filament_yolo26s_per_image_metrics.csv` — per-image metrics

### Resource usage

| Stage | GPU | CPU |
|---|---|---|
| YOLO training | Primary | 4 data workers + OS overhead |
| Validation inference | Primary | 4 metric workers |
| Submission inference | Primary | 4 RLE/post-processing workers |

The custom worker pool intentionally targets four workers × one CPU thread each, so approximately **400% aggregate CPU** is available when the CPU stage is saturated.

`cache="disk"` is used for the YOLO dataset to reduce repeated JPEG decoding and disk I/O during training/validation.